# 11. BERT — Masked Language Model

**Цель:** Реализовать BERT-подобную модель (Encoder-only), обучить на задаче Masked Language Model (MLM), визуализировать контекстуальные представления.

---

In [ ]:
# 80/10/10 masking: 80% [MASK], 10% random, 10% unchanged.
# Why not 100% [MASK]? Model learns only [MASK] patterns, fails to generalize.
import sys, os, logging, math
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("bert")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Using device: %s", device)

## 11.1 Специальные токены BERT

| Токен | ID | Назначение |
|-------|----|------------|
| [PAD] | 0  | Выравнивание длин |
| [CLS] | 1  | Классификационный токен (начало) |
| [SEP] | 2  | Разделитель предложений |
| [MASK]| 3  | Маскированный токен |
| [UNK] | 4  | Неизвестный токен |

In [ ]:
# Special tokens structure input: [CLS] for classification, [SEP] for separation.
PAD, CLS, SEP, MASK, UNK = 0, 1, 2, 3, 4
log.debug("Special tokens defined: PAD=%d, CLS=%d, SEP=%d, MASK=%d, UNK=%d", PAD, CLS, SEP, MASK, UNK)

## 11.2 BERT-архитектура (Encoder-only)

BERT — это стопка энкодерных блоков трансформера с bidirectionnal self-attention (в отличие от GPT, где attention каузальный).

## BERT: почему encoder-only (а не decoder) для понимания

BERT использует MLM (Masked Language Model) — двунаправленный контекст, где модель видит
все токены одновременно. Это принципиально отличается от CLM (Causal Language Model) в GPT,
где внимание каузальное (unidirectional).

| Характеристика | BERT (Encoder) | GPT (Decoder) |
|---|---|---|
| Pre-training | MLM — маскировка 15% токенов | CLM — предсказание следующего токена |
| Attention | Bidirectional (оба направления) | Unidirectional (только влево) |
| Контекст | Полный контекст с обеих сторон | Только левый контекст |
| Задачи | Понимание (NLI, QA, классификация) | Генерация (text completion, chat) |

Для задач **понимания** (NLP understanding) — таких как классификация, NER, QA —
необходим доступ к полному контексту. Encoder-only архитектура BERT позволяет модели
видеть все токены последовательности одновременно, что критически важно для понимания смысла.

> **Ссылка:** Devlin et al., 2018 — *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*


In [ ]:
# BERT is encoder-only: full bidirectional context required for understanding tasks.
# Lower layers capture surface patterns; higher layers capture semantic meaning.
log.debug("Implementing BERT model")

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)
        n_heads = self.W_O.out_features // self.d_k
        Q = self.W_Q(Q).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.W_O.out_features)
        return self.W_O(output)

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = x + self.dropout1(self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask))
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x

class BERT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, num_layers=4, max_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout_pe = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, dropout) for _ in range(num_layers)
        ])
        
        # MLM head
# Linear projection from d_model to vocab_size predicts masked token IDs.
        self.mlm_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, vocab_size),
        )
        
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        
        log.info("BERT: vocab=%d, d_model=%d, n_heads=%d, layers=%d, params=%d",
                 vocab_size, d_model, n_heads, num_layers, sum(p.numel() for p in self.parameters()))
    
    def forward(self, x, mask=None):
        x = self.dropout_pe(self.embedding(x) * math.sqrt(self.d_model) + self.pe[:, :x.size(1), :])
        for layer in self.layers:
            x = layer(x, mask)
        return self.mlm_head(x)  # (batch, seq_len, vocab)

## 11.3 Masked Language Model: маскировка 15% токенов

**Стратегия маскировки (BERT原文):**
- 80% → заменить на [MASK]
- 10% → заменить на случайный токен
- 10% → оставить как есть

## 80/10/10 стратегия маскировки: почему не 100%

В оригинальной статье BERT используется **80/10/10** стратегия для masked tokens:
- **80%** — заменить на `[MASK]`
- **10%** — заменить на случайный токен из словаря
- **10%** — оставить как есть (исходный токен)

### Проблема 100% маскировки

Если всегда заменять выбранные токены на `[MASK]`, модель научится предсказывать
только токены `[MASK]`. Но на этапе fine-tuning токен `[MASK]` **никогда не встречается** —
возникает mismatch между pre-training и fine-tuning.

### Как 80/10/10 решает проблему

- **10% случайных токенов** — модель учится исправлять ошибки, не полагаясь на наличие `[MASK]`.
- **10% неизменённых** — модель учится сохранять исходный токен и понимать, что токен НЕ был заменён.

Этот трюк заставляет модель строить **осмысленные контекстуальные представления**, а не просто
детектировать наличие `[MASK]`.

> **Ссылка:** Devlin et al., 2018 — Section 3.3.1


In [ ]:
# 80/10/10 mitigates pre-train/fine-tune mismatch: model sees non-[MASK] tokens too.
log.debug("Implementing MLM masking function")

def mask_tokens(input_ids, vocab_size, mask_prob=0.15, special_tokens=(PAD, CLS, SEP)):
    """Маскировка токнов для MLM."""
    labels = input_ids.clone()
    
    # Не маскируем специальные токены
    special_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    for tok in special_tokens:
        special_mask = special_mask | (input_ids == tok)
    
    probability = torch.full(input_ids.shape, mask_prob)
    masked = torch.bernoulli(probability).bool()
    masked = masked & ~special_mask  # не трогаем специальные токены
    
    # 80% -> [MASK]
    mask_replace = torch.bernoulli(torch.full(input_ids.shape, 0.8)).bool() & masked
    input_ids[mask_replace] = MASK
    
    # 10% -> random token
    random_replace = torch.bernoulli(torch.full(input_ids.shape, 0.5)).bool() & masked & ~mask_replace
    random_tokens = torch.randint(5, vocab_size, input_ids.shape)
    input_ids[random_replace] = random_tokens[random_replace]
    
    # 10% -> unchanged (labels already set)
    
    # Labels: -100 для позиций, где loss не считаем
    labels[~masked] = -100
    
    return input_ids, labels

# Демонстрация
example = torch.tensor([[CLS, 5, 12, 8, 3, 15, SEP, PAD, PAD]])
masked, labels = mask_tokens(example.clone(), vocab_size=20)
print(f"Original: {example[0].tolist()}")
print(f"Masked:   {masked[0].tolist()}")
print(f"Labels:   {labels[0].tolist()}")
log.info("MLM masking demonstration complete")

## 11.4 Подготовка данных и обучение BERT

In [ ]:
# MLM head: linear d_model -> vocab_size, CE loss on masked positions only.
log.info("Preparing MLM training data")

vocab_size = 30
max_len = 12

def make_mlm_data(num_samples):
    data = []
    for _ in range(num_samples):
        length = np.random.randint(3, max_len - 1)
        seq = [CLS] + np.random.randint(5, vocab_size, size=length).tolist() + [SEP]
        seq = seq + [PAD] * (max_len - len(seq))
        data.append(seq)
    return torch.tensor(data)

train_data = make_mlm_data(2000)
val_data = make_mlm_data(500)

bert = BERT(vocab_size=vocab_size, d_model=32, n_heads=4, num_layers=4, max_len=max_len).to(device)
optimizer = torch.optim.AdamW(bert.parameters(), lr=0.001, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")
print(f"BERT params: {sum(p.numel() for p in bert.parameters()):,}")

In [ ]:
# Encoder-only: bidirectional attention provides rich context for MLM.
log.info("Starting BERT MLM training")

n_epochs = 40
batch_size = 64
best_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    bert.train()
    epoch_loss = 0
    perm = torch.randperm(len(train_data))
    
    for i in range(0, len(train_data), batch_size):
        idx = perm[i:i+batch_size]
        batch = train_data[idx].clone().to(device)
        masked, labels = mask_tokens(batch, vocab_size)
        
        output = bert(masked)
        loss = criterion(output.reshape(-1, vocab_size), labels.reshape(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # Validation
    bert.eval()
    val_loss = 0
    with torch.no_grad():
        for i in range(0, len(val_data), batch_size):
            batch = val_data[i:i+batch_size].clone().to(device)
            masked, labels = mask_tokens(batch, vocab_size)
            output = bert(masked)
            val_loss += criterion(output.reshape(-1, vocab_size), labels.reshape(-1)).item()
    
    avg_train = epoch_loss / (len(train_data) / batch_size)
    avg_val = val_loss / (len(val_data) / batch_size)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    
    if epoch % 10 == 0:
        log.info("Epoch %d: train=%.4f, val=%.4f", epoch, avg_train, avg_val)
    
    if avg_val < best_loss:
        best_loss = avg_val

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BERT MLM Training')
plt.grid(True)
plt.show()
log.info("BERT training complete, best val loss=%.4f", best_loss)

## 11.5 Демонстрация: предсказание замаскированных токенов

In [ ]:
# Contextual embeddings: shallow layers encode syntax; deep layers encode semantics.
log.debug("Predicting masked tokens")

@torch.no_grad()
def predict_masked(model, input_ids):
    model.eval()
    output = model(input_ids.to(device))
    probs = F.softmax(output, dim=-1)
    preds = output.argmax(-1)
    return preds, probs

# Создаём пример с [MASK]
example = torch.tensor([[CLS, 12, 7, MASK, 19, 5, SEP, PAD, PAD, PAD, PAD, PAD]])
preds, probs = predict_masked(bert, example)

# Показываем предсказание для маскированной позиции
mask_pos = (example == MASK).nonzero()[0, 1].item()
mask_probs = probs[0, mask_pos]
top5 = mask_probs.argsort(descending=True)[:5]

print(f"Input: {example[0].tolist()}")
print(f"Prediction at position {mask_pos}:")
for tok in top5:
    print(f"  Token {tok.item():2d}: {mask_probs[tok].item():.2%}")
log.info("MLM prediction demo complete")

## 11.6 Контекстуальные представления разных слоёв

Извлечём эмбеддинги из разных слоёв и покажем, как они меняются.

In [ ]:
log.debug("Extracting contextual embeddings from all layers")
# Lower layers: local text patterns (surface). Higher layers: global meaning (semantics).

class BERTWithHooks(BERT):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.layer_outputs = []
    
    def forward(self, x, mask=None):
        self.layer_outputs = []
        x = self.dropout_pe(self.embedding(x) * math.sqrt(self.d_model) + self.pe[:, :x.size(1), :])
        self.layer_outputs.append(x.detach())
        for layer in self.layers:
            x = layer(x, mask)
            self.layer_outputs.append(x.detach())
        return self.mlm_head(x)

bert_hooks = BERTWithHooks(vocab_size=vocab_size, d_model=32, n_heads=4, num_layers=4, max_len=max_len).to(device)
bert_hooks.load_state_dict(bert.state_dict())

test_seq = torch.tensor([[CLS, 10, 15, 8, 12, 6, 18, SEP, PAD, PAD, PAD, PAD]])
with torch.no_grad():
    bert_hooks(test_seq.to(device))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, ax in enumerate(axes.flat):
    if i < len(bert_hooks.layer_outputs):
        im = ax.imshow(bert_hooks.layer_outputs[i][0].cpu().numpy(), cmap='viridis', aspect='auto')
        ax.set_title(f'Layer {i}' if i > 0 else 'Embedding')
        ax.set_xlabel('d_model')
        ax.set_ylabel('Token')
        plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Contextual Embeddings Through BERT Layers')
plt.tight_layout()
plt.show()
log.info("Contextual embedding visualization complete")

In [ ]:
# BERT encoder-only architecture learned bidirectional representations via MLM.
print("=== BERT (Masked Language Model) complete ===")
print("Topics covered:")
print("  - Special tokens: [CLS], [SEP], [MASK], [PAD]")
print("  - BERT architecture (Encoder-only)")
print("  - MLM masking strategy (80/10/10)")
print("  - BERT MLM training loop")
print("  - Predicting masked tokens")
print("  - Contextual embeddings through layers")
print(f"  - Best MLM loss: {best_loss:.4f}")
log.info("BERT notebook complete")

## Полезные ссылки

- **BERT paper:** Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. arXiv:1810.04805.
  - [arXiv](https://arxiv.org/abs/1810.04805)
- **MLM Objective:** Section 3.1 — Masked Language Model.
- **80/10/10 Strategy:** Section 3.3.1 — Masking details.
- **Contextual Representations:** Layer-wise analysis in Section 5.4 and Appendix C.
